In [1]:
import os
import pandas as pd
import numpy as np
import scipy.signal as signal
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import load_model
import joblib

2025-04-04 12:59:28.868208: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-04 12:59:28.869073: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-04-04 12:59:28.875990: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-04-04 12:59:28.894693: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1743760768.924657   19539 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1743760768.93

In [3]:


# === Paths ===
base_path = '../data/kaggle-drdataboston/'
encoder = load_model(os.path.join(base_path, 'encoder_10steps_with_gyro.keras'))
scaler = joblib.load(os.path.join(base_path, 'scaler_10steps_with_gyro.pkl'))
model_gender = load_model(os.path.join(base_path, 'predict_gender.keras'))
model_age = load_model(os.path.join(base_path, 'predict_age.keras'))
model_weight = load_model(os.path.join(base_path, 'predict_weight.keras'))
model_height = load_model(os.path.join(base_path, 'predict_height.keras'))

# === Main Function ===
def predict_from_dataframe(df):
    fs_target = 50  # 50 Hz target

    required_cols = [
        'timestamp',
        'motionUserAccelerationX.G.', 'motionUserAccelerationY.G.', 'motionUserAccelerationZ.G.',
        'gyroRotationX.rad.s.', 'gyroRotationY.rad.s.', 'gyroRotationZ.rad.s.'
    ]
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"Missing required column: {col}")

    # Convert timestamp to datetime if needed
    if not np.issubdtype(df['timestamp'].dtype, np.datetime64):
        df['timestamp'] = pd.to_datetime(df['timestamp'])

    df = df.sort_values('timestamp')
    df = df.set_index('timestamp').resample('20ms').mean().interpolate()  # 50Hz

    acc_mag = np.sqrt(
        df['motionUserAccelerationX.G.']**2 +
        df['motionUserAccelerationY.G.']**2 +
        df['motionUserAccelerationZ.G.']**2
    )

    # Bandpass filter 0.5–3Hz
    b, a = signal.butter(2, [0.5 / (fs_target / 2), 3.0 / (fs_target / 2)], btype='band')
    acc_filt = signal.filtfilt(b, a, acc_mag)

    # Peak detection
    peaks, _ = signal.find_peaks(acc_filt, distance=fs_target * 0.4)

    step_windows = []
    for i in range(len(peaks) - 10):
        start = peaks[i]
        end = peaks[i + 10]

        if end - start < 30:
            continue

        window = df.iloc[start:end][[
            'motionUserAccelerationX.G.',
            'motionUserAccelerationY.G.',
            'motionUserAccelerationZ.G.',
            'gyroRotationX.rad.s.',
            'gyroRotationY.rad.s.',
            'gyroRotationZ.rad.s.'
        ]].to_numpy()

        resampled = signal.resample(window, 200)
        flat = resampled.flatten()
        step_windows.append(flat)

    if not step_windows:
        raise ValueError("No valid 10-step segments found in data.")

    X = np.array(step_windows)
    X_scaled = scaler.transform(X)
    X_encoded = encoder.predict(X_scaled)

    # Predict for each batch
    age_preds = model_age.predict(X_encoded).flatten()
    weight_preds = model_weight.predict(X_encoded).flatten()
    height_preds = model_height.predict(X_encoded).flatten()
    gender_probs = model_gender.predict(X_encoded).flatten()

    return {
        'Gender': 'Female' if gender_probs.mean() > 0.5 else 'Male',
        'Gender Confidence': round(gender_probs.mean(), 3),
        'Age': round(age_preds.mean(), 1),
        'Weight': round(weight_preds.mean(), 1),
        'Height': round(height_preds.mean(), 1),
        'Num Segments Used': len(X_encoded)
    }

   

In [10]:
import pandas as pd
import numpy as np

num = 5

folder_path = f'../data/mendeley-Cristiana-Gabriel/datasets/{num}'
# === Load files ===
acc = pd.read_csv(f"{folder_path}/accelerometer.txt", sep=r'\s+', engine='python')
gyro = pd.read_csv(f"{folder_path}/gyroscope.txt", sep=r'\s+', engine='python')

# === Rename columns ===
acc.columns = ['timestamp', 'acc_x', 'acc_y', 'acc_z']
gyro.columns = ['timestamp', 'gyro_x', 'gyro_y', 'gyro_z']

# === Convert to datetime and set index ===
acc['timestamp'] = pd.to_datetime(acc['timestamp'], unit='ms')
gyro['timestamp'] = pd.to_datetime(gyro['timestamp'], unit='ms')

acc.set_index('timestamp', inplace=True)
gyro.set_index('timestamp', inplace=True)

# === Resample to 50Hz ===
acc_50hz = acc.resample('20ms').mean().interpolate()
gyro_50hz = gyro.resample('20ms').mean().interpolate()

# === Merge ACC and Gyro ===
merged = pd.merge(acc_50hz, gyro_50hz, left_index=True, right_index=True, how='inner')

# === Final formatting ===
merged.reset_index(inplace=True)
merged.rename(columns={
    'timestamp': 'timestamp',
    'acc_x': 'motionUserAccelerationX.G.',
    'acc_y': 'motionUserAccelerationY.G.',
    'acc_z': 'motionUserAccelerationZ.G.',
    'gyro_x': 'gyroRotationX.rad.s.',
    'gyro_y': 'gyroRotationY.rad.s.',
    'gyro_z': 'gyroRotationZ.rad.s.'
}, inplace=True)

# === Save or use directly ===
merged.to_csv(f"../data/kaggle-drdataboston/test_data/{num}.csv", index=False)
print(f"✅ Saved combined data to {num}.csv")


✅ Saved combined data to 5.csv


In [5]:
from scipy.signal import butter, filtfilt

def filter_acceleration(df, fs=50):
    acc_cols = ['motionUserAccelerationX.G.', 'motionUserAccelerationY.G.', 'motionUserAccelerationZ.G.']
    filtered = df.copy()
    b, a = butter(2, 0.5 / (fs / 2), btype='highpass')  # Cutoff = 0.5Hz
    for col in acc_cols:
        filtered[col] = filtfilt(b, a, filtered[col])
    return filtered

In [9]:
test_num = 2

test_file = os.path.join(base_path, f'test_data/{test_num}.csv')  # should include timestamp, acc, gyro
df = pd.read_csv(test_file)
df = filter_acceleration(df)
result = predict_from_dataframe(df)
print("\nPredicted Profile:")
for key, val in result.items():
    print(f" - {key}: {val}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step

Predicted Profile:
 - Gender: Male
 - Gender Confidence: 0.3619999885559082
 - Predicted Age: 22.100000381469727
 - Predicted Weight: 64.80000305175781
 - Predicted Height: 170.60000610351562
 - Num Segments Used: 21


In [13]:
import os

base_input = '../data/mendeley-Cristiana-Gabriel/datasets'
base_output = '../data/kaggle-drdataboston/test_data'

for num in os.listdir(base_input):
    folder_path = os.path.join(base_input, num)
    if not os.path.isdir(folder_path):
        continue

    try:
        acc = pd.read_csv(f"{folder_path}/accelerometer.txt", sep=r'\s+', engine='python')
        gyro = pd.read_csv(f"{folder_path}/gyroscope.txt", sep=r'\s+', engine='python')

        acc.columns = ['timestamp', 'acc_x', 'acc_y', 'acc_z']
        gyro.columns = ['timestamp', 'gyro_x', 'gyro_y', 'gyro_z']

        acc['timestamp'] = pd.to_datetime(acc['timestamp'], unit='ms')
        gyro['timestamp'] = pd.to_datetime(gyro['timestamp'], unit='ms')

        acc.set_index('timestamp', inplace=True)
        gyro.set_index('timestamp', inplace=True)

        acc_50hz = acc.resample('20ms').mean().interpolate()
        gyro_50hz = gyro.resample('20ms').mean().interpolate()

        merged = pd.merge(acc_50hz, gyro_50hz, left_index=True, right_index=True, how='inner')
        merged.reset_index(inplace=True)
        merged.rename(columns={
            'timestamp': 'timestamp',
            'acc_x': 'motionUserAccelerationX.G.',
            'acc_y': 'motionUserAccelerationY.G.',
            'acc_z': 'motionUserAccelerationZ.G.',
            'gyro_x': 'gyroRotationX.rad.s.',
            'gyro_y': 'gyroRotationY.rad.s.',
            'gyro_z': 'gyroRotationZ.rad.s.'
        }, inplace=True)

        os.makedirs(base_output, exist_ok=True)
        merged.to_csv(os.path.join(base_output, f"{num}.csv"), index=False)
        print(f"✅ Saved combined data to {num}.csv")
    except Exception as e:
        print(f"❌ Failed for {num}: {e}")


✅ Saved combined data to 98.csv
✅ Saved combined data to 138.csv
✅ Saved combined data to 76.csv
✅ Saved combined data to 166.csv
✅ Saved combined data to 26.csv
✅ Saved combined data to 202.csv
✅ Saved combined data to 97.csv
✅ Saved combined data to 65.csv
✅ Saved combined data to 5.csv
✅ Saved combined data to 132.csv
✅ Saved combined data to 174.csv
✅ Saved combined data to 171.csv
✅ Saved combined data to 103.csv
✅ Saved combined data to 22.csv
✅ Saved combined data to 27.csv
✅ Saved combined data to 58.csv
✅ Saved combined data to 135.csv
✅ Saved combined data to 180.csv
✅ Saved combined data to 206.csv
✅ Saved combined data to 94.csv
✅ Saved combined data to 144.csv
✅ Saved combined data to 162.csv
✅ Saved combined data to 1.csv
✅ Saved combined data to 141.csv
✅ Saved combined data to 85.csv
✅ Saved combined data to 91.csv
✅ Saved combined data to 167.csv
✅ Saved combined data to 119.csv
✅ Saved combined data to 60.csv
✅ Saved combined data to 8.csv
✅ Saved combined data to 161

In [5]:
import os
import json
import pandas as pd
from sklearn.metrics import mean_absolute_error

base_path = '../data/kaggle-drdataboston'
input_base = '../data/mendeley-Cristiana-Gabriel/datasets'

predictions = []
ground_truths = []

for filename in os.listdir(os.path.join(base_path, 'test_data')):
    if not filename.endswith('.csv'):
        continue

    test_num = filename.replace('.csv', '')
    test_file = os.path.join(base_path, f'test_data/{filename}')
    df = pd.read_csv(test_file)
    df = filter_acceleration(df)
    result = predict_from_dataframe(df)

    json_path = os.path.join(input_base, test_num, 'user_data.json')
    if not os.path.exists(json_path):
        continue

    with open(json_path) as f:
        gt = json.load(f)

    predictions.append({
        'age': result['Age'],
        'weight': result['Weight'],
        'height': result['Height'],
        'gender': 1 if result['Gender'] == 'Male' else 0
    })
    ground_truths.append({
        'age': gt['age'],
        'weight': gt['weight_kg'],
        'height': gt['height_cm'],
        'gender': 1 if gt['genre'] == 'Male' else 0
    })

# Convert to DataFrame
pred_df = pd.DataFrame(predictions)
gt_df = pd.DataFrame(ground_truths)

# Compute MAE for each
mae = (pred_df - gt_df).abs().mean()
print("\n📊 Mean Absolute Errors:")
for col in mae.index:
    print(f" - {col}: {mae[col]:.2f}")


2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/2 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/stepWARNING:tensorflow:6 out of the last 6 calls to <function TensorFlowTrainer.make_predict_function.<locals>.one_step_on_data_distributed at 0x735d7ff40310> triggered tf.function retracing. Tracing is expensive and the excessive number of tracings could be due to (1) creating @tf.function repeatedly in a loop, (2) passing tensors with different shapes, (3) passing Python objects instead of tensors. For (1), please define your @tf.function outside of the loop. For (2), @tf.function has reduce_retracing=True option that can avoid unnecessary retracing. For (3), please refer to https://www.tensorflow.org/guide/function#controlling_retracing and https://www.tensorflow.org/api_docs/python/tf/function for  more details.
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
1/1 ━━━━━━━━━

In [7]:
import json
import pandas as pd

def load_and_prepare_json(filepath):
    # === Step 1: Load and parse JSON ===
    with open(filepath, 'r') as f:
        data = json.load(f)

    payload = data['payload']
    interval_s = payload['interval_ms'] / 1000  # convert ms to seconds
    values = payload['values']

    # === Step 2: Construct DataFrame ===
    timestamps = pd.date_range(start='2024-01-01', periods=len(values), freq=f'{int(interval_s * 1000)}ms')
    df = pd.DataFrame(values, columns=[
        'motionUserAccelerationX.G.',
        'motionUserAccelerationY.G.',
        'motionUserAccelerationZ.G.'
    ])
    df['timestamp'] = timestamps

    # === Step 3: Add dummy gyro data (zero-filled) ===
    df['gyroRotationX.rad.s.'] = 0.0
    df['gyroRotationY.rad.s.'] = 0.0
    df['gyroRotationZ.rad.s.'] = 0.0

    # === Step 4: Filter acceleration ===
    df = filter_acceleration(df)

    return df




In [12]:
import os

location = '../data/personal/'
fname = 'SebiColt'
start = 1
end = 1

results = []

for i in range(start, end + 1):
    filename = f'{fname}{i}.json'
    df = load_and_prepare_json(os.path.join(location, filename))
    result = predict_from_dataframe(df)
    results.append(result)

    print(f"\nPredicted Profile for {filename}:")
    for key, val in result.items():
        print(f" - {key}: {val}")

# === Calculate the mean of numeric fields ===
numeric_keys = ['Age', 'Weight', 'Height', 'Gender Confidence', 'Num Segments Used']
mean_profile = {key: round(sum(r[key] for r in results) / len(results), 2) for key in numeric_keys}

# For gender, majority vote
gender_votes = [r['Gender'] for r in results]
mean_profile['Gender'] = max(set(gender_votes), key=gender_votes.count)

with open(f'../data/personal/predict-{fname}.json', 'w') as f:
    print("\n Mean Predicted Profile:")
    f.write(f"Mean Predicted profile from {end - start + 1} files:\n")
    for key, val in mean_profile.items():
        print(f" - {key}: {val}")
        f.write(f" - {key}: {val}\n")



1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step

Predicted Profile for SebiColt1.json:
 - Gender: Female
 - Gender Confidence: 0.6949999928474426
 - Age: 22.700000762939453
 - Weight: 57.29999923706055
 - Height: 173.39999389648438
 - Num Segments Used: 16

 Mean Predicted Profile:
 - Age: 22.7
 - Weight: 57.3
 - Height: 173.4
 - Gender Confidence: 0.69
 - Num Segments Used: 16.0
 - Gender: Female
